In [1]:
import torch 
from torch import nn
import pandas as pd
import os
os.environ["CUDA_VISIBLE_DEVICES"]='0'
# from alphagen.config import *
# from alphagen.data.tokens import *
from alphagen.models.alpha_pool import AlphaPoolBase, AlphaPool
from alphagen.rl.env.core import AlphaEnvCore
import torch.nn.functional as F
from gan.dataset import Collector
from gan.network.generater import NetG_DCGAN
from gan.network.masker import NetM
from gan.network.predictor import NetP, train_regression_model,train_regression_model_with_weight
from alphagen.rl.env.wrapper import SIZE_ACTION,action2token

from alphagen_generic.features import open_
from gan.utils import Builders
from alphagen_generic.features import *
from alphagen.data.expression import *

from gan.utils.data import get_data_by_year


In [2]:
instruments: str = "csi300"
freq = 'day'
save_name = 'aff_baseline'
window = float('inf')

In [ ]:
from alphagen.utils.correlation import batch_pearsonr,batch_spearmanr
device = 'cuda:0'
result = []
pred_dfs = {}

root_path = os.path.expanduser('~/out/AlphaForge/')

for n_factors in [10, 20]:
    for seed in [0, 42, 128, 256, 512]:
        cur_seed_ic = []
        cur_seed_ric = []
        all_pred_df_list = []
        # for train_end in range(2020,2021):
        train_end = '2023-06-30'
        print(n_factors,seed,train_end)
        
        returned = get_data_by_year(
            train_start = '2013-07-01',train_end = '2023-06-30',
            valid_start = '2023-07-01', valid_end = '2024-06-30',
            test_start = '2024-07-01', test_end = '2025-06-30',
            instruments=instruments, target=target, freq=freq,
        )
        data_all, data,data_valid,data_valid_withhead,data_test,data_test_withhead,name = returned
        

        path = f'{root_path}/{save_name}_{instruments}_{train_end}_{seed}/z_bld_zoo_final.pkl'
        tensor_save_path = f'{root_path}/{save_name}_{instruments}_{train_end}_{seed}/pred_{train_end}_{n_factors}_{window}_{seed}.pt'

        pred = torch.load(tensor_save_path).to(device)
        tgt = target.evaluate(data_all)
        
        
        ones = torch.ones_like(tgt)
        ones = ones * torch.nan
        ones[-data_test.n_days:] = pred
        cur_df = data_all.make_dataframe(ones)
        all_pred_df_list.append(cur_df.unstack().iloc[-data_test.n_days:].stack())
        
        tgt = tgt[-data_test.n_days:].to(device)
        
        
        ic_s = torch.nan_to_num(batch_pearsonr(pred,tgt),nan=0)
        rank_ic_s = torch.nan_to_num(batch_spearmanr(pred,tgt),nan=0)

        cur_seed_ic.append(ic_s)
        cur_seed_ric.append(rank_ic_s)
            
        pred_dfs[f"{n_factors}_{seed}"] = pd.concat(all_pred_df_list,axis=0)
        ic = torch.cat(cur_seed_ic)
        rank_ic = torch.cat(cur_seed_ric)

        ic_mean = ic.mean().item()
        rank_ic_mean = rank_ic.mean().item()
        ic_std = ic.std().item()
        rank_ic_std = rank_ic.std().item()
        tmp = dict(
            seed = seed,
            num = n_factors,
            ic = ic_mean,
            ric = rank_ic_mean,
            icir = ic_mean/ic_std,
            ricir = rank_ic_mean/rank_ic_std,
        )
        result.append(tmp)

10 0 2023-06-30


/tmp/ipykernel_196254/3912945719.py:37: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  all_pred_df_list.append(cur_df.unstack().iloc[-data_test.n_days:].stack())


10 42 2023-06-30


/tmp/ipykernel_196254/3912945719.py:37: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  all_pred_df_list.append(cur_df.unstack().iloc[-data_test.n_days:].stack())


10 128 2023-06-30


...

/tmp/ipykernel_196254/3912945719.py:37: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  all_pred_df_list.append(cur_df.unstack().iloc[-data_test.n_days:].stack())


20 512 2023-06-30


/tmp/ipykernel_196254/3912945719.py:37: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  all_pred_df_list.append(cur_df.unstack().iloc[-data_test.n_days:].stack())


20 1024 2023-06-30


/tmp/ipykernel_196254/3912945719.py:37: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  all_pred_df_list.append(cur_df.unstack().iloc[-data_test.n_days:].stack())


In [6]:
import pandas as pd
run_result = pd.DataFrame(result).groupby(['num','seed']).mean().groupby('num').agg(['mean','std'])
print(run_result)

           ic                 ric                icir              ricir  \
         mean       std      mean       std      mean      std      mean   
num                                                                        
10   0.017211  0.012158  0.039763  0.015984  0.082159  0.05936  0.194207   
20   0.017211  0.012158  0.039763  0.015984  0.082159  0.05936  0.194207   

               
          std  
num            
10   0.084392  
20   0.084392  


In [8]:
result

[{'seed': 0,
  'num': 10,
  'ic': 0.022852333262562752,
  'ric': 0.058048538863658905,
  'icir': 0.1076383081405432,
  'ricir': 0.28105611235862316},
 {'seed': 42,
  'num': 10,
  'ic': 0.012365691363811493,
  'ric': 0.04497420787811279,
  'icir': 0.05570932139887511,
  'ricir': 0.21205234801881553},
 {'seed': 128,
  'num': 10,
  'ic': 0.009938284754753113,
  'ric': 0.03265558183193207,
  'icir': 0.04479197811989838,
  'ricir': 0.14281182395483258},
 {'seed': 256,
  'num': 10,
  'ic': 0.03224692866206169,
  'ric': 0.03984290361404419,
  'icir': 0.14712499270852297,
  'ricir': 0.18956828919179883},
 {'seed': 512,
  'num': 10,
  'ic': -0.0006373617798089981,
  'ric': 0.012426290661096573,
  'icir': -0.002911203135875079,
  'ricir': 0.06055627236509921},
 {'seed': 1024,
  'num': 10,
  'ic': 0.026499086990952492,
  'ric': 0.05062802881002426,
  'icir': 0.14059963806530257,
  'ricir': 0.2791952010752368},
 {'seed': 0,
  'num': 20,
  'ic': 0.022852333262562752,
  'ric': 0.058048538863658905,


In [9]:
import json

with open(f'{root_path}/{save_name}_{instruments}_results.json','w') as f:
    json.dump(result,f)